In [8]:
import os, glob, gc, math, sqlite3
from types import SimpleNamespace
import numpy as np
import pandas as pd

# ------------------------------- settings -------------------------------
DB_PATH = '/content/drive/MyDrive/route_all.db'
ROUTES = None             # e.g. [56, 70, 506] = only these routes
LAST_DAYS = None         # use only the last N service days (None = all days)

MAX_AHEAD = 10            # how many stops ahead to make pairs for
TABLE = None              # e.g. 'journeys'              (None = the table with the most rows)
OUT_DIR = 'csv.gz tables'            # set a folder name to also save the two csv.gz tables (slow for a big dataset; None = skip)

# --- memory control (this is what keeps the kernel alive) ---
TRAIN_FRAC = None         # None = automatic: take as many training rows as fit in memory (all of them when they fit).
                          # Or a number like 0.3 = use a random 30% of the training rows. The test day is NEVER sampled.
RAM_BUDGET_GB = None      # None = automatic: the training table may use 45% of the free memory. Or e.g. 6.
MAX_ROWS = 50_000_000     # stop early if the database filter still matches more rows than this
CHUNK = 500_000           # rows read from the database at a time

NEEDED = ['trip_id', 'stop_position', 'is_realtime', 'scheduled_departure_sec',
          'delay_seconds', 'route', 'stop_id']
OPTIONAL = ['service_date', 'polled_at', 'pattern_code']
LABELS = ['trip_id', 'route', 'stop_id', 'pattern_code']

BASE = ['stop_position_now', 'sched_now', 'hour', 'weekday', 'is_rush_hour', 'stops_in_trip',
        'delay_now', 'prev_delay_now', 'trip_first_delay', 'delay_mean3', 'delay_mean5', 'trend3',
        'stop_position_target', 'stops_ahead', 'sched_gap', 'delay_trend', 'hour_target',
        'ahead_delay_target', 'ahead_age', 'ahead_change',
        'hist_target_hour', 'hist_now_hour', 'hist_target_stop', 'hist_now_stop',
        'hist_change_hour', 'hist_change_stop']



def free_ram():
    """Bytes of memory this notebook can still use (looks at the container limit too)."""
    try:
        import psutil
        avail = psutil.virtual_memory().available
    except Exception:
        avail = 8 * 2**30
        try:
            for line in open('/proc/meminfo'):
                if line.startswith('MemAvailable'):
                    avail = int(line.split()[1]) * 1024
        except Exception:
            pass
    try:
        lim = open('/sys/fs/cgroup/memory.max').read().strip()
        if lim != 'max':
            avail = min(avail, int(lim) - int(open('/sys/fs/cgroup/memory.current').read()))
    except Exception:
        pass
    return int(avail)


class Codes:
    """Turns labels (trip ids, routes, stops ...) into small integers, the same way for every chunk.
    A trip id as text costs ~100 bytes per row; as an integer it costs 4."""
    def __init__(self):
        self.maps = {}

    def encode(self, name, s):
        m = self.maps.setdefault(name, {})
        codes, uniques = pd.factorize(s)
        ids = np.array([m.setdefault(u, len(m)) for u in uniques], dtype=np.int32)
        out = np.full(len(codes), -1, dtype=np.int32)
        ok = codes >= 0
        out[ok] = ids[codes[ok]]
        return out

    def names(self):
        return {k: [str(x) for x in sorted(m, key=m.get)] for k, m in self.maps.items()}


def to_dates(sd):
    return pd.to_datetime(sd.astype(str), format='%Y%m%d') if pd.api.types.is_numeric_dtype(sd) else pd.to_datetime(sd)


def prepare(df, codes):
    """Makes a table small: only the needed columns, labels -> int32 codes, numbers -> 32 bit, dates -> datetime.
    Works on one chunk of the database or on a whole DataFrame."""
    df = df[[c for c in df.columns if c in NEEDED + OPTIONAL]].copy().reset_index(drop=True)
    df = df.dropna(subset=['stop_position', 'scheduled_departure_sec'])
    if 'service_date' not in df:
        df['service_date'] = pd.to_datetime(df['trip_id'].astype(str).str.extract(r'_(\d{8})_')[0], format='%Y%m%d')
    else:
        df['service_date'] = to_dates(df['service_date'])
    for c in LABELS:
        if c in df:
            df[c] = codes.encode(c, df[c])
    df = df[df['trip_id'] >= 0]
    r = df['is_realtime']
    if r.dtype != bool:
        df['is_realtime'] = (r != 0) if pd.api.types.is_numeric_dtype(r) else r.astype(str).str.lower().isin(['1', 'true', 't', 'yes', '1.0'])
    df['stop_position'] = df['stop_position'].astype(np.int32)
    df['scheduled_departure_sec'] = df['scheduled_departure_sec'].astype(np.float32)   # seconds: exact in 32 bit
    df['delay_seconds'] = df['delay_seconds'].astype(np.float32)
    if 'polled_at' in df and not (pd.api.types.is_numeric_dtype(df['polled_at']) or pd.api.types.is_datetime64_any_dtype(df['polled_at'])):
        df['polled_at'] = pd.to_datetime(df['polled_at'])
    return df.reset_index(drop=True)

def find_db():
    files = [f for ext in ('db', 'sqlite', 'sqlite3') for f in glob.glob(f'*.{ext}') + glob.glob(f'/content/*.{ext}')]
    if not files:
        raise SystemExit('no .db file found: set DB_PATH at the top')
    return files[0]


def load_table(db_path, codes, table=None, last_days=None, routes=None):
    """Reads only what is needed, CHUNK rows at a time, and shrinks every chunk before the next one is read."""
    with sqlite3.connect(db_path) as con:
        names = pd.read_sql("SELECT name FROM sqlite_master WHERE type IN ('table','view') AND name NOT LIKE 'sqlite_%'", con)['name']
        sizes = {n: int(pd.read_sql(f'SELECT COUNT(*) AS n FROM "{n}"', con)['n'][0]) for n in names}
        print('tables in the file:', sizes)
        if not sizes:
            raise SystemExit('no tables found (is this really a SQLite file?)')
        table = table or max(sizes, key=sizes.get)
        cols = list(pd.read_sql(f'SELECT * FROM "{table}" LIMIT 1', con).columns)
        missing = [c for c in NEEDED if c not in cols]
        if missing:
            raise SystemExit(f'MISSING columns: {missing}\ncolumns found: {cols}\n'
                             '-> tell me these names and the cleaning can be adjusted')
        keep = [c for c in cols if c in NEEDED + OPTIONAL]
        where = ["(is_realtime = 1 OR LOWER(CAST(is_realtime AS TEXT)) IN ('1', 'true', 't', 'yes'))"]
        if routes:
            where.append('CAST(route AS TEXT) IN (' + ', '.join(repr(str(r)) for r in routes) + ')')
        if last_days and 'service_date' in cols:
            days = pd.read_sql(f'SELECT DISTINCT service_date AS d FROM "{table}" ORDER BY d DESC LIMIT {int(last_days)}', con)['d']
            first = days.min()
            print('using the last', len(days), 'service days:', first, 'to', days.max())
            where.append('service_date >= ' + (repr(first) if isinstance(first, str) else str(int(first))))
        sql = 'SELECT ' + ', '.join(f'"{c}"' for c in keep) + f' FROM "{table}" WHERE ' + ' AND '.join(where)
        n = int(pd.read_sql('SELECT COUNT(*) AS n FROM (' + sql + ')', con)['n'][0])
        print(f'rows that will be loaded: {n:,}')
        if n == 0:
            r = pd.read_sql(f'SELECT DISTINCT route FROM "{table}" LIMIT 30', con)['route'].tolist()
            raise SystemExit(f'0 rows match. Routes in the table (first 30): {r}. Check ROUTES / LAST_DAYS at the top.')
        if n > MAX_ROWS:
            raise SystemExit(f'{n:,} rows is more than MAX_ROWS ({MAX_ROWS:,}). Set ROUTES and/or a smaller LAST_DAYS at the top.')
        parts, done = [], 0
        for chunk in pd.read_sql(sql, con, chunksize=CHUNK):
            parts.append(prepare(chunk, codes))
            done += len(chunk)
            print(f'  read {done:,} / {n:,}', end='\r')
            del chunk
        print()
    df = pd.concat(parts, ignore_index=True)
    del parts; gc.collect()
    print(f'using table "{table}": {len(df):,} rows, {df.shape[1]} columns, {df.memory_usage().sum() / 2**20:,.0f} MB')
    return df


def standardize(df, codes):
    """Adds what is missing: a polling order and the pattern (= one route in one direction = route + the stop it starts from)."""
    if 'polled_at' not in df:
        df['polled_at'] = np.arange(len(df), dtype=np.int64)
        print('no polled_at column: using the row order of the table instead')
    if 'pattern_code' not in df:
        first = df.loc[df.groupby('trip_id')['stop_position'].idxmin().to_numpy(), ['trip_id', 'route', 'stop_id']]
        rn, sn = codes.names()['route'], codes.names()['stop_id']
        pair = first['route'].to_numpy(np.int64) * (len(sn) + 1) + first['stop_id'].to_numpy(np.int64)
        pcode, uniq = pd.factorize(pair)
        label = [f'{rn[u // (len(sn) + 1)]}_{sn[u % (len(sn) + 1)]}' for u in uniq]
        codes.maps['pattern_code'] = {lab: i for i, lab in enumerate(label)}
        by_trip = np.full(int(df['trip_id'].max()) + 1, -1, dtype=np.int32)
        by_trip[first['trip_id'].to_numpy()] = pcode
        df['pattern_code'] = by_trip[df['trip_id'].to_numpy()]
        print('no pattern_code column: made one from route + first stop ->', len(label), 'patterns',
              '(expect about 2 per route; many more means trips that start mid-way)')
    return df.drop(columns=[c for c in ['stop_id'] if c in df])


def clean_raw(df):
    df = df.sort_values('polled_at', kind='stable').drop_duplicates(['trip_id', 'stop_position'], keep='last')
    df = df[df['is_realtime']].drop(columns=['polled_at', 'is_realtime'])
    df = df.sort_values(['trip_id', 'stop_position']).reset_index(drop=True)

    df['hour'] = (df['scheduled_departure_sec'] // 3600).clip(upper=23).astype(np.int8)
    df['weekday'] = df['service_date'].dt.dayofweek.astype(np.int8)
    df['is_rush_hour'] = df['hour'].isin([7, 8, 9, 15, 16, 17]).astype(np.int8)

    g = df.groupby('trip_id', sort=False)
    df['prev_delay'] = g['delay_seconds'].shift(1)
    df['trip_first_delay'] = g['delay_seconds'].transform('first')
    df['stops_in_trip'] = (g['stop_position'].transform('max') + 1).astype(np.int32)
    del g

    df = df.dropna(subset=['prev_delay'])
    lo, hi = df['delay_seconds'].quantile([0.001, 0.999])
    df['is_outlier'] = ((df['delay_seconds'] < lo) | (df['delay_seconds'] > hi)).astype(np.int8)

    keep = ['service_date', 'trip_id', 'route', 'pattern_code', 'stop_position',
            'scheduled_departure_sec', 'hour', 'weekday', 'is_rush_hour', 'stops_in_trip',
            'prev_delay', 'trip_first_delay', 'is_outlier', 'delay_seconds']
    return df[keep].sort_values('service_date', kind='stable').reset_index(drop=True)


def build_pairs(df, names, max_ahead=15, max_ahead_age=1800, train_frac=None, ram_budget=None, seed=0):
    """Returns the training and test tables as float32 numpy matrices (not DataFrames).
    The test day is the last service day. Each day is built on its own and written straight into a
    preallocated matrix, so only about one day of working copies exists at any time."""
    days = [pd.Timestamp(x) for x in sorted(df['service_date'].unique())]
    if len(days) < 2:
        raise SystemExit('need at least 2 service days (the last one is the test day)')
    test_day, train_days = days[-1], days[:-1]
    P = int(df['stop_position'].max()) + 1
    n_route, n_pat = len(names['route']), len(names['pattern_code'])
    features = BASE + ['route_code', 'pattern_code']
    nf, nb = len(features), len(BASE)

    def k3(pat, stop, hour):
        return (pat.astype(np.int64) * P + stop.astype(np.int64)) * 24 + hour.astype(np.int64)

    def k2(pat, stop):
        return pat.astype(np.int64) * P + stop.astype(np.int64)


    day_rows, agg3, agg2 = {}, {}, {}
    for d, sub in df.groupby('service_date', sort=True):
        d = pd.Timestamp(d)
        day_rows[d] = len(sub)
        if d != test_day:
            dl = pd.Series(sub['delay_seconds'].to_numpy(np.float64))
            pat, stp, hr = sub['pattern_code'].to_numpy(), sub['stop_position'].to_numpy(), sub['hour'].to_numpy()
            agg3[d] = dl.groupby(k3(pat, stp, hr)).agg(['sum', 'count'])
            agg2[d] = dl.groupby(k2(pat, stp)).agg(['sum', 'count'])
    tot3 = pd.concat(list(agg3.values())).groupby(level=0).sum()
    tot2 = pd.concat(list(agg2.values())).groupby(level=0).sum()


    row_bytes = 4 * nf + 12
    full_train = sum(day_rows[d] for d in train_days) * max_ahead
    test_pairs = day_rows[test_day] * max_ahead
    budget = (ram_budget if ram_budget else 0.45 * free_ram()) - test_pairs * row_bytes
    if train_frac is None:
        train_frac = float(min(1.0, max(0.005, budget / (full_train * row_bytes))))
    print(f'columns per row: {nf} | {n_route} routes, {n_pat} patterns (one-hot only inside the models) | about {row_bytes} bytes per pair')
    print(f'training pairs if everything is used: up to {full_train:,} = {full_train * row_bytes / 2**30:.1f} GB '
          f'| free memory now: {free_ram() / 2**30:.1f} GB')
    if test_pairs * row_bytes > 0.7 * free_ram():
        raise SystemExit(f'the test day alone ({test_pairs:,} pairs, {test_pairs * row_bytes / 2**30:.1f} GB) does not fit in memory. '
                         'Set ROUTES (fewer routes) or MAX_AHEAD (smaller) at the top, or use a machine with more RAM.')
    if train_frac < 1:
        print(f'>>> using a random {train_frac:.1%} of the training rows so the kernel stays alive '
              f'(set RAM_BUDGET_GB / TRAIN_FRAC at the top to change this). The test day is complete.')
    else:
        print('>>> all training rows fit in memory')
    take = {d: min(day_rows[d], int(round(train_frac * day_rows[d]))) for d in train_days}
    take[test_day] = day_rows[test_day]

    cap_tr = sum(take[d] for d in train_days) * max_ahead
    cap_te = take[test_day] * max_ahead
    X_tr, A_tr = np.empty((cap_tr, nf), np.float32), np.empty((cap_tr, 3), np.float32)
    X_te, A_te = np.empty((cap_te, nf), np.float32), np.empty((cap_te, 3), np.float32)
    used = {'tr': 0, 'te': 0}


    for d, sub in df.groupby('service_date', sort=True):
        d = pd.Timestamp(d)
        is_test = d == test_day
        sub = sub.sort_values(['trip_id', 'stop_position'], kind='stable').reset_index(drop=True)
        w = pd.DataFrame({
            'trip_id': sub['trip_id'].to_numpy(np.int64), 'pattern_code': sub['pattern_code'].to_numpy(),
            'route': sub['route'].to_numpy(), 'stop_position': sub['stop_position'].to_numpy(np.int64),
            'sched': sub['scheduled_departure_sec'].to_numpy(np.float64), 'delay': sub['delay_seconds'].to_numpy(np.float64),
            'hour': sub['hour'].to_numpy(np.int64), 'weekday': sub['weekday'].to_numpy(np.int64),
            'is_rush_hour': sub['is_rush_hour'].to_numpy(np.int64), 'stops_in_trip': sub['stops_in_trip'].to_numpy(np.int64),
            'prev_delay': sub['prev_delay'].to_numpy(np.float64), 'trip_first_delay': sub['trip_first_delay'].to_numpy(np.float64)})
        del sub
        w['actual'] = w['sched'] + w['delay']
        tr = w['trip_id']

        cs = w['delay'].fillna(0).groupby(tr).cumsum()
        cn = w['delay'].notna().astype(np.int64).groupby(tr).cumsum()
        for k in (3, 5):
            s = cs - cs.groupby(tr).shift(k).fillna(0)
            c = cn - cn.groupby(tr).shift(k).fillna(0)
            w[f'delay_mean{k}'] = s / c.where(c > 0)
        w['trend3'] = w.groupby('trip_id', sort=False)['delay'].diff(3) / 3.0
        del cs, cn

        now = pd.DataFrame({
            'trip_id': w['trip_id'], 'pattern_code': w['pattern_code'], 'route': w['route'],
            'stop_position_now': w['stop_position'], 'sched_now': w['sched'], 'actual_now': w['actual'],
            'hour': w['hour'], 'weekday': w['weekday'], 'is_rush_hour': w['is_rush_hour'], 'stops_in_trip': w['stops_in_trip'],
            'delay_now': w['delay'], 'prev_delay_now': w['prev_delay'], 'trip_first_delay': w['trip_first_delay'],
            'delay_mean3': w['delay_mean3'], 'delay_mean5': w['delay_mean5'], 'trend3': w['trend3']})
        if take[d] < len(now):
            now = now.sample(n=take[d], random_state=seed)
        tgt = pd.DataFrame({'trip_id': w['trip_id'], 'stop_position_target': w['stop_position'],
                            'sched_target': w['sched'], 'delay_target': w['delay']})

        pairs = pd.concat([now.merge(tgt.assign(stop_position_now=tgt['stop_position_target'] - k), on=['trip_id', 'stop_position_now'])
                           for k in range(1, max_ahead + 1)], ignore_index=True)
        del now, tgt
        pairs['stops_ahead'] = pairs['stop_position_target'] - pairs['stop_position_now']
        pairs['sched_gap'] = pairs['sched_target'] - pairs['sched_now']
        pairs['delay_trend'] = pairs['delay_now'] - pairs['prev_delay_now']
        pairs['hour_target'] = (pairs['sched_target'] // 3600).clip(upper=23)
        pairs['delay_change'] = pairs['delay_target'] - pairs['delay_now']


        right = pd.DataFrame({'pattern_code': w['pattern_code'], 'stop_position_target': w['stop_position'],
                              'ahead_actual_target': w['actual'], 'ahead_delay_target': w['delay'],
                              'ahead_trip': w['trip_id']}).dropna(subset=['ahead_actual_target']).sort_values('ahead_actual_target')
        pairs = pairs.dropna(subset=['actual_now']).sort_values('actual_now').reset_index(drop=True)
        pairs = pd.merge_asof(pairs, right, left_on='actual_now', right_on='ahead_actual_target',
                              by=['pattern_code', 'stop_position_target'], direction='backward')
        pairs['ahead_age'] = pairs['actual_now'] - pairs['ahead_actual_target']
        bad = (pairs['ahead_trip'] == pairs['trip_id']) | (pairs['ahead_age'] > max_ahead_age)
        pairs.loc[bad, ['ahead_delay_target', 'ahead_age']] = np.nan
        pairs['ahead_trip'] = pairs['ahead_trip'].where(~bad).fillna(-1).astype(np.int64)
        lookup = pd.DataFrame({'ahead_trip': w['trip_id'], 'stop_position_now': w['stop_position'],
                               'ahead_delay_at_now_stop': w['delay']})
        pairs = pairs.merge(lookup, on=['ahead_trip', 'stop_position_now'], how='left')
        pairs['ahead_change'] = pairs['ahead_delay_target'] - pairs['ahead_delay_at_now_stop']
        del right, lookup, w


        t3 = tot3 if is_test else tot3.sub(agg3[d], fill_value=0)
        t2 = tot2 if is_test else tot2.sub(agg2[d], fill_value=0)
        h3 = t3['sum'] / t3['count'].where(t3['count'] > 0)
        h2 = t2['sum'] / t2['count'].where(t2['count'] > 0)
        pat = pairs['pattern_code'].to_numpy()
        sp_t, sp_n = pairs['stop_position_target'].to_numpy(), pairs['stop_position_now'].to_numpy()
        pairs['hist_target_hour'] = h3.reindex(k3(pat, sp_t, pairs['hour_target'].to_numpy())).to_numpy()
        pairs['hist_now_hour'] = h3.reindex(k3(pat, sp_n, pairs['hour'].to_numpy())).to_numpy()
        pairs['hist_target_stop'] = h2.reindex(k2(pat, sp_t)).to_numpy()
        pairs['hist_now_stop'] = h2.reindex(k2(pat, sp_n)).to_numpy()
        pairs['hist_change_hour'] = pairs['hist_target_hour'] - pairs['hist_now_hour']
        pairs['hist_change_stop'] = pairs['hist_target_stop'] - pairs['hist_now_stop']


        X, A, key = (X_te, A_te, 'te') if is_test else (X_tr, A_tr, 'tr')
        a, m = used[key], len(pairs)
        for j, c in enumerate(BASE):
            X[a:a + m, j] = pairs[c].to_numpy()
        X[a:a + m, nb] = pairs['route'].to_numpy()
        X[a:a + m, nb + 1] = pat
        A[a:a + m] = pairs[['sched_target', 'delay_change', 'delay_target']].to_numpy(np.float32)
        used[key] = a + m
        del pairs
        gc.collect()
        print(f'  {d.date()}  {"TEST" if is_test else "train"}  {m:>10,} pairs', end='\r')
    print()

    X_tr, A_tr, X_te, A_te = X_tr[:used['tr']], A_tr[:used['tr']], X_te[:used['te']], A_te[:used['te']]
    return SimpleNamespace(features=features, n_base=nb, n_route=n_route, n_pat=n_pat, X_train=X_tr, A_train=A_tr, X_test=X_te, A_test=A_te,
                           test_now=X_te[:, BASE.index('delay_now')].copy(), test_true=A_te[:, 2].astype(np.float64),
                           test_stops_ahead=X_te[:, BASE.index('stops_ahead')].copy(),
                           train_days=[str(x.date()) for x in train_days], test_day=str(test_day.date()), train_frac=train_frac)


def save_tables(P, folder, chunk=200_000):
    """Writes the pair tables to csv.gz piece by piece (never the whole table in memory)."""
    os.makedirs(folder, exist_ok=True)
    cols = P.features + ['sched_target', 'delay_change', 'delay_target']
    for name, X, A in [('pairs_train', P.X_train, P.A_train), ('pairs_test', P.X_test, P.A_test)]:
        path = f'{folder}/{name}.csv.gz'
        if os.path.exists(path):
            os.remove(path)
        for i in range(0, len(X), chunk):
            pd.DataFrame(np.hstack([X[i:i + chunk], A[i:i + chunk]]), columns=cols).to_csv(
                path, index=False, mode='a', header=(i == 0), compression='gzip')
        print('saved', path)



def fill_and_scale(X_train, X_test, n_cols, step=200_000, sample=500_000, seed=0):
    """Fills empty values with the TRAINING medians and scales with the TRAINING mean/spread, IN PLACE
    (a few hundred thousand rows at a time), so the big matrix is never copied. Only the first n_cols columns are
    touched (the route / pattern code columns at the end stay as they are).
    The medians come from a random sample of up to `sample` training rows."""
    rng = np.random.default_rng(seed)
    pick = np.sort(rng.choice(len(X_train), min(len(X_train), sample), replace=False))
    import warnings
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        med = np.nan_to_num(np.nanmedian(X_train[pick, :n_cols], axis=0), nan=0.0).astype(np.float32)
    s, ss = np.zeros(n_cols), np.zeros(n_cols)
    for i in range(0, len(X_train), step):
        c = X_train[i:i + step, :n_cols]
        np.copyto(c, med, where=np.isnan(c))
        c64 = c.astype(np.float64)
        s += c64.sum(0); ss += (c64 * c64).sum(0)
    mean = s / len(X_train)
    std = np.sqrt(np.maximum(ss / len(X_train) - mean ** 2, 0.0))
    std[std < 1e-6] = 1.0
    mean32, std32 = mean.astype(np.float32), std.astype(np.float32)
    for X in (X_train, X_test):
        for i in range(0, len(X), step):
            c = X[i:i + step, :n_cols]
            if X is X_test:
                np.copyto(c, med, where=np.isnan(c))
            c -= mean32; c /= std32
    return med, mean32, std32


def expand_np(Xc, nb, n_route, n_pat):
    """Rows of the big matrix -> the full design matrix: base columns, one-hot route, one-hot pattern, and a 1 for the intercept."""
    m = len(Xc)
    out = np.zeros((m, nb + n_route + n_pat + 1))
    out[:, :nb] = Xc[:, :nb]
    rows = np.arange(m)
    r, p = Xc[:, nb].astype(np.int64), Xc[:, nb + 1].astype(np.int64)
    out[rows[r >= 0], nb + r[r >= 0]] = 1
    out[rows[p >= 0], nb + n_route + p[p >= 0]] = 1
    out[:, -1] = 1
    return out


def fit_linear(X, y, nb, n_route, n_pat, step=20_000):
    """Ordinary least squares (same answer as sklearn's LinearRegression) from running sums, a few rows at a time."""
    d = nb + n_route + n_pat + 1
    A, b = np.zeros((d, d)), np.zeros(d)
    for i in range(0, len(X), step):
        c = expand_np(X[i:i + step], nb, n_route, n_pat)
        A += c.T @ c; b += c.T @ y[i:i + step]
    return np.linalg.lstsq(A, b, rcond=None)[0]


def predict_linear(w, X, nb, n_route, n_pat, step=20_000):
    return np.concatenate([expand_np(X[i:i + step], nb, n_route, n_pat) @ w for i in range(0, len(X), step)])


In [9]:
import sys
def say(*a): print(*a); sys.stdout.flush()

codes = Codes()
if 'df' in globals() and isinstance(df, pd.DataFrame):
    raw = df
    say('using the DataFrame df already in memory:', f'{len(raw):,} rows')
    if ROUTES:
        raw = raw[raw['route'].astype(str).isin([str(r) for r in ROUTES])]
    if LAST_DAYS and 'service_date' in raw:
        keep_days = sorted(to_dates(raw['service_date']).unique())[-LAST_DAYS:]
        raw = raw[to_dates(raw['service_date']).isin(keep_days)]
        say('kept the last', LAST_DAYS, 'days:', f'{len(raw):,} rows')
    raw = prepare(raw, codes)                         # a small copy: only the needed columns, 32-bit numbers, integer labels
else:
    db_path = DB_PATH or find_db()
    say('database:', db_path)
    raw = load_table(db_path, codes, TABLE, LAST_DAYS, ROUTES)

say('1/3 standardize ...')
raw = standardize(raw, codes)
say('2/3 clean ...')
clean = clean_raw(raw)
del raw; gc.collect()
say(f'cleaned rows: {len(clean):,} | service days: {clean["service_date"].nunique()}')

say('3/3 build pairs (the big step: one service day at a time) ...')
pairs = build_pairs(clean, codes.names(), max_ahead=MAX_AHEAD, train_frac=TRAIN_FRAC,
                    ram_budget=RAM_BUDGET_GB * 2**30 if RAM_BUDGET_GB else None)     # test = the last day; train = every earlier day
del clean; gc.collect()
say(f'train {len(pairs.X_train):,} rows | test {len(pairs.X_test):,} rows | training days {pairs.train_days} | test day {pairs.test_day}')

if OUT_DIR:
    save_tables(pairs, OUT_DIR)


database: /content/drive/MyDrive/route_all.db
tables in the file: {'stoptimes': 9069343}
rows that will be loaded: 3,637,310
  read 3,637,310 / 3,637,310
using table "stoptimes": 3,637,310 rows, 8 columns, 114 MB
1/3 standardize ...
no polled_at column: using the row order of the table instead
no pattern_code column: made one from route + first stop -> 711 patterns (expect about 2 per route; many more means trips that start mid-way)
2/3 clean ...
cleaned rows: 3,518,577 | service days: 8
3/3 build pairs (the big step: one service day at a time) ...
columns per row: 28 | 362 routes, 711 patterns (one-hot only inside the models) | about 124 bytes per pair
training pairs if everything is used: up to 33,477,660 = 3.9 GB | free memory now: 9.6 GB
>>> all training rows fit in memory
  2026-10-17  TEST   1,391,343 pairs
train 27,300,966 rows | test 1,391,343 rows | training days ['2026-10-10', '2026-10-11', '2026-10-12', '2026-10-13', '2026-10-14', '2026-10-15', '2026-10-16'] | test day 2026-

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
import time, json
import torch
from torch import nn
import torch.nn.functional as F
from sklearn.metrics import mean_absolute_error, r2_score
from xgboost import XGBRegressor

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

FEATURES = pairs.features
print(len(FEATURES), 'input features')
X_all, X_test = pairs.X_train, pairs.X_test
y_all = pairs.A_train[:, 1].copy()
test_now, test_true, test_stops_ahead = pairs.test_now, pairs.test_true, pairs.test_stops_ahead

NB, NR, NP = pairs.n_base, pairs.n_route, pairs.n_pat
D_IN = NB + NR + NP
print(f'network input width: {D_IN} ({NB} numeric + {NR} routes + {NP} patterns)')
medians, mean_, std_ = fill_and_scale(X_all, X_test, NB)
print(f'training matrix: {X_all.shape[0]:,} rows x {X_all.shape[1]} columns = {X_all.nbytes / 2**30:.2f} GB')


on_gpu = device == 'cuda' and X_all.nbytes < 0.5 * torch.cuda.mem_get_info()[0]
data_dev = device if on_gpu else 'cpu'
X_tr = torch.from_numpy(X_all).to(data_dev)
y_tr = torch.from_numpy(y_all).view(-1, 1).to(data_dev)
X_te = torch.from_numpy(X_test)
print('training data lives on:', data_dev)


def expand(xb):
    """One batch of rows -> network input. The route and pattern codes become one-hot HERE, a batch at a time,
    so the huge one-hot table is never stored."""
    base = xb[:, :NB]
    r = xb[:, NB].long() + 1
    p = xb[:, NB + 1].long() + 1
    return torch.cat([base, F.one_hot(r, NR + 1)[:, 1:].to(base.dtype), F.one_hot(p, NP + 1)[:, 1:].to(base.dtype)], 1)


def batch(X, y, idx):
    return expand(X[idx].to(device)), y[idx].to(device)


@torch.inference_mode()
def predict(model, X, bs=100_000):
    """Predictions in pieces, so the test day never needs one huge intermediate tensor."""
    model.eval()
    return torch.cat([model(expand(X[i:i + bs].to(device))).cpu() for i in range(0, len(X), bs)]).numpy().ravel()


@torch.inference_mode()
def mae_on(model, X, y, bs=100_000):
    model.eval(); tot = 0.0
    for i in range(0, len(X), bs):
        tot += (model(expand(X[i:i + bs].to(device))) - y[i:i + bs].to(device)).abs().sum().item()
    return tot / len(X)


results, preds = {}, {}

def score(name, change):
    """change = predicted CHANGE in delay. Add delay_now back, then score against the real delay at stop B."""
    p = test_now.astype(np.float64) + np.asarray(change, dtype=np.float64)
    preds[name] = p
    results[name] = (mean_absolute_error(test_true, p), r2_score(test_true, p))
    print(f'{name:38s} test MAE {results[name][0]:6.2f} s   R2 {results[name][1]:.3f}')

score('1. copy the current delay', np.zeros(len(test_now)))


device: cuda
28 input features
network input width: 1099 (26 numeric + 362 routes + 711 patterns)
training matrix: 27,300,966 rows x 28 columns = 2.85 GB
training data lives on: cuda
1. copy the current delay              test MAE  44.91 s   R2 0.777


In [11]:
w_lin = fit_linear(X_all, y_all, NB, NR, NP)      # same answer as LinearRegression, computed a few rows at a time
change_lin = predict_linear(w_lin, X_test, NB, NR, NP)
score('2. linear regression', change_lin)


def xgb_frame(X):
    """Numeric columns + route and pattern as categorical columns (XGBoost splits on them directly: no one-hot table)."""
    d = pd.DataFrame(X[:, :NB], columns=pairs.features[:NB])
    d['route_code'] = pd.Categorical.from_codes(X[:, NB].astype(np.int64), categories=np.arange(NR))
    d['pattern_code'] = pd.Categorical.from_codes(X[:, NB + 1].astype(np.int64), categories=np.arange(NP))
    return d

xgb_params = dict(n_estimators=300, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8,
                  objective='reg:absoluteerror', tree_method='hist', n_jobs=-1, random_state=0)
try:
    xgb = XGBRegressor(**xgb_params, enable_categorical=True, max_bin=256).fit(xgb_frame(X_all), y_all)
    change_xgb = xgb.predict(xgb_frame(X_test))
except Exception as e:                                # an older XGBoost without categorical support: use the codes as plain numbers
    print('categorical XGBoost not available (', type(e).__name__, ') -> using route/pattern codes as numbers')
    xgb = XGBRegressor(**xgb_params, max_bin=128).fit(X_all, y_all)
    change_xgb = xgb.predict(X_test)
score('3. XGBoost', change_xgb)


2. linear regression                   test MAE  36.02 s   R2 0.858



KeyboardInterrupt



In [12]:
class Bus_Bet_A(nn.Module):
    def __init__(self, d_in):
        super().__init__()
        self.layer_1 = nn.Linear(d_in, 40)
        self.layer_2 = nn.Linear(40, 10)
        self.layer_3 = nn.Linear(10, 1)
        self.relu = nn.ReLU()
    def forward(self, x):
        out = self.relu(self.layer_1(x))
        out = self.relu(self.layer_2(out))
        return self.layer_3(out)


torch.manual_seed(0)
model_a = Bus_Bet_A(D_IN).to(device)
loss_fn = nn.L1Loss()
opt = torch.optim.Adam(model_a.parameters(), lr=0.01)
n, bs = len(X_tr), 1024

for epoch in range(10):
    t0 = time.time()
    model_a.train()
    perm = torch.randperm(n, device=X_tr.device)
    total = torch.zeros((), device=device)
    for i in range(0, n, bs):
        idx = perm[i:i + bs]
        xb, yb = batch(X_tr, y_tr, idx)
        loss = loss_fn(model_a(xb), yb)
        opt.zero_grad(); loss.backward(); opt.step()
        total += loss.detach() * len(idx)
    print(f"epoch {epoch} | train loss {total.item() / n:.4f} | {time.time() - t0:.0f} s")

change_a = predict(model_a, X_te)
score('4. neural network A (40-10)', change_a)


epoch 0 | train loss 34.3524 | 56 s
epoch 1 | train loss 33.4903 | 53 s
epoch 2 | train loss 33.2725 | 53 s
epoch 3 | train loss 33.1305 | 53 s
epoch 4 | train loss 33.0444 | 56 s
epoch 5 | train loss 32.9838 | 51 s
epoch 6 | train loss 32.9393 | 51 s
epoch 7 | train loss 32.9028 | 52 s
epoch 8 | train loss 32.8725 | 52 s
epoch 9 | train loss 32.8469 | 52 s
4. neural network A (40-10)            test MAE  31.49 s   R2 0.883


In [15]:
class Bus_Bet(nn.Module):
    def __init__(self, d_in):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, 128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 64),  nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 1),
        )
    def forward(self, x):
        return self.net(x)


torch.manual_seed(5)
model = Bus_Bet(D_IN).to(device)
loss_fn = nn.L1Loss()
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-3)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=2)

n_val = int(0.1 * len(X_tr))
Xa, ya = X_tr[:-n_val], y_tr[:-n_val]
Xv, yv = X_tr[-n_val:], y_tr[-n_val:]

best, best_state, bs = 1e9, None, 1024
for epoch in range(15):
    t0 = time.time()
    model.train()
    perm = torch.randperm(len(Xa), device=Xa.device)
    for i in range(0, len(Xa), bs):
        idx = perm[i:i + bs]
        xb, yb = batch(Xa, ya, idx)
        loss = loss_fn(model(xb), yb)
        opt.zero_grad(); loss.backward(); opt.step()
    v = mae_on(model, Xv, yv)
    sched.step(v)
    if v < best:
        best, best_state = v, {k: t.detach().clone() for k, t in model.state_dict().items()}
    print(f"epoch {epoch} | val MAE {v:.3f} | {time.time() - t0:.0f} s")

model.load_state_dict(best_state)
change_b = predict(model, X_te)
score('5. neural network B (128-64, dropout)', change_b)


epoch 0 | val MAE 33.924 | 52 s
epoch 1 | val MAE 33.660 | 50 s
epoch 2 | val MAE 33.461 | 49 s
epoch 3 | val MAE 33.413 | 49 s
epoch 4 | val MAE 33.392 | 49 s
epoch 5 | val MAE 33.338 | 49 s
epoch 6 | val MAE 33.284 | 49 s
epoch 7 | val MAE 33.243 | 50 s
epoch 8 | val MAE 33.250 | 49 s
epoch 9 | val MAE 33.438 | 50 s
epoch 10 | val MAE 33.314 | 50 s
epoch 11 | val MAE 33.311 | 50 s
epoch 12 | val MAE 33.324 | 49 s
epoch 13 | val MAE 33.288 | 49 s
epoch 14 | val MAE 33.268 | 49 s
5. neural network B (128-64, dropout)  test MAE  31.26 s   R2 0.883


In [18]:
table = pd.DataFrame(results, index=['test MAE (s)', 'R2']).T.round(3).sort_values('test MAE (s)')
table['better than copying the delay by (s)'] = (results['1. copy the current delay'][0] - table['test MAE (s)']).round(2)
display(table) if 'display' in globals() else print(table.to_string())

by_dist = pd.DataFrame({'stops_ahead': test_stops_ahead.astype(int),
                        'copy delay': np.abs(preds['1. copy the current delay'] - test_true),
                        'linear': np.abs(preds['2. linear regression'] - test_true),

                        'network B': np.abs(preds['5. neural network B (128-64, dropout)'] - test_true),

                       }).groupby('stops_ahead').mean().round(1)
print('\nmean miss in seconds by how many stops ahead:')
print(by_dist.T.to_string())

                                       test MAE (s)     R2  better than copying the delay by (s)
5. neural network B (128-64, dropout)        31.255  0.883                                 13.65
4. neural network A (40-10)                  31.495  0.883                                 13.41
2. linear regression                         36.018  0.858                                  8.89
1. copy the current delay                    44.908  0.777                                 -0.00

mean miss in seconds by how many stops ahead:
stops_ahead    1     2     3     4     5     6     7     8     9     10
copy delay   25.0  31.9  37.6  42.4  46.6  50.2  53.1  56.1  58.5  60.7
linear       18.0  24.2  29.1  33.4  37.1  40.6  43.6  46.3  48.9  51.1
network B    14.4  20.5  25.0  28.9  32.3  35.3  38.1  40.7  43.2  45.4
